# Schema Definition with `TypedDict`

When building agentic LangGraph applications, you define a **state schema** so every node in your graph knows exactly what keys and types the shared state contains. This is about **short-term memory and context management** — the data passed between nodes during a run.

For **long-term memory**, agents often use vector databases, relational databases, or LangGraph's own persistence features (checkpointers, thread history, and the store API). Those systems survive beyond a single graph invocation.

## What is `TypedDict`?

`TypedDict` is a Python type hint. Type hints let you define the exact **schema** of a dictionary before you use it.

For example, you might define a dictionary with one key `graph_state` of type `str`. That helps **you and your team** keep graph code consistent as the application grows.

Think of it like TypeScript: you define an `interface` or `type`, then pass it to functions so IDEs and type checkers understand what each node expects.

> **Important:** `TypedDict` is **not** runtime validation. It does not stop bad LLM output by itself. If an LLM returns `"1984"` instead of `1984`, Python will not raise an error unless *you* add separate validation (for example with Pydantic or structured output APIs).

## Static typing vs runtime behavior

| Layer | What happens on a schema mismatch? |
|-------|-------------------------------------|
| **IDE / type checker** (Pyright, mypy) | **Warning or error** while you write code — squiggly underlines, problems panel |
| **Python at runtime** | **Nothing** — TypedDict is ignored; the code still runs |

So yes: your IDE will usually **warn** you about mismatches, but Python itself will **not** raise an error when the notebook or script executes.

To validate LLM responses at runtime, use tools designed for that job — structured output, JSON schema enforcement, or Pydantic models.

In [ ]:
from typing import TypedDict


class MyGraph(TypedDict):
    graph_state: str


# Your IDE / type checker should flag this — but Python still runs it.
node1: MyGraph = {
    "graph_state": 1984,  # type: ignore[typeddict-item]  # remove ignore to see checker warning
}

node1

## Declaring a typed dictionary

Once you define a schema class, you annotate variables with it:

```python
from typing import TypedDict

class MyGraph(TypedDict):
    graph_state: str

node1: MyGraph = {
    "graph_state": "active"
}
```

## Optional keys

By default, all `TypedDict` keys are **required** — every instance must include them.

Three patterns matter:

1. **Key may be missing** → `NotRequired[...]`
2. **Most keys optional, a few required** → `class State(TypedDict, total=False)` plus `Required[...]` on mandatory keys
3. **Key must exist, value may be `None`** → `documents: list[str] | None`

In LangGraph, nodes usually return **partial updates** (only the keys they changed). Optional keys and reducers (covered below) make that pattern work cleanly.

In [ ]:
from typing import Literal, NotRequired, Required, TypedDict


class ExampleState(TypedDict, total=False):
    graph_state: Required[str]  # required even with total=False
    documents: NotRequired[list[str]]  # key may be absent
    domain: NotRequired[Literal["finance", "legal", "marketing"]]


# Valid: only required key present
minimal: ExampleState = {"graph_state": "active"}
minimal

## Modern Python type hints (no import needed)

For simple variables, modern Python (3.9+) lets you use built-in generics directly — no `typing` import required:

In [ ]:
book_list: list[str] = ["The Hobbit", "1984"]
unique_codes: set[int] = {501, 404, 200}
product_prices: dict[str, float] = {"laptop": 999.999, "mouse": 25.50}

book_list, unique_codes, product_prices

## LangGraph: defining graph state

When building with **LangGraph**, import `TypedDict` from `typing` to define your state schema. This gives every node a shared contract for what lives in state.

Pattern:

1. Import `TypedDict` (and helpers like `Annotated`, `NotRequired`, `Literal`)
2. Define your TypeScript-like interface as a class
3. Pass the type hint into node functions so your IDE knows what `state` contains
4. Return **only the keys you want to update** from each node — LangGraph merges the partial result into the existing state

## State reducers with `Annotated`

By default, LangGraph **overwrites** a state key when a node returns it. If Node A returns `{"documents": ["doc2"]}`, the previous `documents` list is replaced.

For keys like `messages`, you usually want to **append**, not replace. Attach a reducer with `Annotated`:

- **Type slot** (`list`) — tells static checkers what data type to expect
- **Reducer** (`add_messages`) — tells LangGraph how to merge old and new values

LangGraph reads the reducer from the type metadata and applies it during the merge step.

In [ ]:
from typing import Annotated, Literal, NotRequired, TypedDict

from langgraph.graph.message import add_messages


class GraphState(TypedDict):
    graph_state: str
    documents: NotRequired[list[str]]
    messages: Annotated[list, add_messages]
    domain: NotRequired[Literal["finance", "legal", "marketing"]]


def agent_node(state: GraphState):
    # Nodes return partial updates — LangGraph merges them into state.
    # No need to return every key on every call.
    return {"documents": ["doc1", "doc2"]}


initial_state: GraphState = {
    "graph_state": "active",
    "messages": [],
}

agent_node(initial_state)